# 02 - Exploratory data analysis

**Owner:** Yuhan Zhang (Modelling lead)  
**Depends on:** `01_data_check.ipynb`  
**Goal:** decide the final class list and document how we handle Surgery, document-type labels, duplicates, label leakage, missing text, and cross-label contradictions, so the split (Issue #2) can be frozen on clean labels.

Run top to bottom. Works in Google Colab (free tier) or locally. Only edit the CONFIG cell.

## 0. Setup

In [ ]:
import sys
import re
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = Path("/content")
elif Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()

DATA_DIR = ROOT / "data"
print(f"Running in {'Colab' if IN_COLAB else 'local'}; data folder: {DATA_DIR}")

## 1. CONFIG

In [ ]:
KAGGLE_DATASET = "tboyle10/medicaltranscriptions"
CSV_PATH       = DATA_DIR / "mtsamples.csv"

TEXT_COL    = "transcription"
LABEL_COL   = "medical_specialty"

RANDOM_SEED = 42

## 2. Load the data

Same loading logic as `01_data_check.ipynb`, plus a label clean-up step (see the note below).

In [ ]:
def download_from_kaggle():
    """Colab only: download mtsamples.csv using the Kaggle key stored in Colab Secrets."""
    import os
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    get_ipython().system(f"kaggle datasets download -d {KAGGLE_DATASET} -p {DATA_DIR} --unzip")

if not CSV_PATH.exists():
    if IN_COLAB:
        download_from_kaggle()
    else:
        raise FileNotFoundError(
            f"{CSV_PATH} not found. Download mtsamples.csv first, see data/README.md."
        )

df = pd.read_csv(CSV_PATH)

# DATA BUG: every value in medical_specialty has a leading space (e.g. " Surgery").
# Strip it once here; every downstream mapping and split depends on this.
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

print(f"Loaded {len(df):,} rows and {df.shape[1]} columns")
print("Columns:", list(df.columns))
df.head()

## 3. Full label inventory

All 40 raw labels, sorted by count, with cumulative share. Basis for the long-tail threshold.

In [ ]:
counts = df[LABEL_COL].value_counts()
share = (counts / counts.sum() * 100).round(2)
cum = share.cumsum().round(2)

label_table = pd.DataFrame({"count": counts, "share_%": share, "cumulative_%": cum})
label_table.index.name = LABEL_COL
print(f"Number of raw labels: {label_table.shape[0]}")
label_table

## 4. Document-type labels vs specialties

Some labels are document types, not medical specialties. **Decision: drop them.** They do not define a routing destination, and a model that predicts a document type is not useful for the stated task.

In [ ]:
DOC_TYPE_LABELS = [
    "Consult - History and Phy.",
    "SOAP / Chart / Progress Notes",
    "Discharge Summary",
    "Emergency Room Reports",
    "Office Notes",
    "Letters",
]

print("Document-type labels to drop:")
for c in DOC_TYPE_LABELS:
    n = counts.get(c, 0)
    print(f"  {c!r}: {n} rows")
total = counts.reindex(DOC_TYPE_LABELS).fillna(0).astype(int).sum()
print(f"Total dropped by this rule: {total} rows")
print("(one of these rows also has no transcription, so the pipeline in "
      "Section 12 removes 938 in this step)")

## 5. Surgery overlap

Surgery is ~22% of rows. **Decision: drop the Surgery label.** 1,054 of 1,103 Surgery rows are byte-for-byte identical to a note already labelled with another specialty, so Surgery is a procedure type, not a specialty. Of the 49 unique Surgery rows, 28 map to a real specialty (ports, breast biopsies, ERCP, etc.) and are re-labelled via the review step (Section 9); the remaining 21 are general-surgery / wound-care procedures with no specialty home, reviewed by the modelling lead and all confirmed as DROP.

In [ ]:
def normalize_text(s):
    """Lowercase, strip punctuation, collapse whitespace. Used for duplicate matching."""
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["_norm"] = df[TEXT_COL].map(normalize_text)

surgery_mask = df[LABEL_COL].eq("Surgery")
others = df.loc[~surgery_mask]
surgery_raw_dup = df.loc[surgery_mask & df[TEXT_COL].isin(set(others[TEXT_COL]))]

print(f"Surgery rows: {surgery_mask.sum()}")
print(f"Byte-for-byte identical to another specialty's note: {len(surgery_raw_dup)}")
print(f"Unique Surgery rows (re-labelled later): {surgery_mask.sum() - len(surgery_raw_dup)}")

## 6. Duplicates

Only ~4% of rows have a unique text; the rest are duplicates, most filed under 2+ labels. **Decision: keep duplicate rows but force the same normalized text into the same split** (removes leakage while keeping data, per the evaluation protocol).

In [ ]:
exact_dup = df[TEXT_COL].duplicated().sum()
norm_dup = df["_norm"].duplicated().sum()
unique_texts = (df["_norm"].value_counts() == 1).sum()

print(f"Exact duplicate rows: {exact_dup} ({exact_dup/len(df):.1%})")
print(f"Normalized duplicate rows: {norm_dup} ({norm_dup/len(df):.1%})")
print(f"Texts appearing exactly once: {unique_texts} ({unique_texts/len(df):.1%})")

## 7. Label leakage

The `keywords` and `description` columns and some note text contain the label itself. **Decision: model on `transcription` only, drop `keywords`/`description`.** The label string also appears inside the transcription in 8.7% of rows, but MTSamples notes are single-line, so header-line stripping would delete the whole note. A masked variant of the text (label strings removed) is planned at modeling time to quantify this residual leakage.

In [ ]:
def label_in_text(col):
    """Count rows whose col value contains the row's own label, case-insensitive.
    Rows with missing text are excluded from the denominator."""
    hit = 0
    n = 0
    for lab, val in zip(df[LABEL_COL], df[col]):
        if not isinstance(val, str):
            continue
        n += 1
        if str(lab).lower() in val.lower():
            hit += 1
    return hit, n

for col in ["keywords", "description"]:
    if col in df.columns:
        hit, n = label_in_text(col)
        print(f"{col}: label string appears in {hit}/{n} rows ({hit/n:.1%})")

# MTSamples transcriptions are single-line, so "first line" equals the whole
# text; this measures the label string appearing anywhere in the note.
def first_line(s):
    if not isinstance(s, str):
        return None
    for line in s.splitlines():
        if line.strip():
            return line.strip()
    return None

df["_first_line"] = df[TEXT_COL].map(first_line)
hit, n = label_in_text("_first_line")
print(f"transcription text: label string appears in {hit}/{n} rows ({hit/n:.1%})")

## 8. Missing or empty text

**Decision: drop rows with missing transcription** (33 rows, 0.7%).

In [ ]:
print(f"Missing transcriptions: {df[TEXT_COL].isna().sum()}")
print(f"Empty transcriptions:   {(df[TEXT_COL].astype(str).str.strip() == '').sum()}")
print(f"Missing labels:         {df[LABEL_COL].isna().sum()}")

## 9. Cross-label contradictions

After dropping document types and Surgery, the same note text is still filed under 2+ *real* specialties (e.g. an angiogram under both Cardiovascular and Radiology). A single-label task requires one label per text.

**Decision: assign one primary specialty per text via a keyword-rule first pass + human review.** The 606-note queue was split into two 303-note batches and reviewed by two teammates in a browser tool; 603 notes were assigned to a class and 3 marked DROP in the first review. Three labels were then flagged by the LLM assistant and corrected after confirmation by the modelling lead (602 assigned, 4 DROP). Section 12 applies the review results.

In [ ]:
DOC_TYPES = DOC_TYPE_LABELS  # defined in Section 4
clean = df[df[TEXT_COL].notna() & ~df[LABEL_COL].isin(DOC_TYPES) & df[LABEL_COL].ne("Surgery")].copy()

g = clean.groupby("_norm").agg(cnt=("sample_name", "size"), nlab=(LABEL_COL, "nunique"))
once = (g["cnt"] == 1).sum()
single_label = (g["nlab"] == 1).sum()
cross = ((g["cnt"] > 1) & (g["nlab"] > 1)).sum()

print(f"After cleaning: {len(clean)} rows")
print(f"Texts appearing exactly once: {once}")
print(f"Single-label texts: {single_label} (kept under one label, even if the row is duplicated)")
print(f"Cross-label contradictory texts: {cross} (resolved via review table)")

## 10. Final class list

Eleven classes (slightly above the 6-10 target, because Psychiatry / Psychology is a distinct specialty with 53 rows and no natural merge target). Rare labels with no natural home are dropped.

| Final class | Raw labels mapped in |
|---|---|
| Cardiovascular / Pulmonary | Cardiovascular / Pulmonary |
| Orthopedic | Orthopedic, Podiatry |
| Radiology | Radiology |
| General Medicine | General Medicine, Pain Management, Physical Medicine - Rehab, Sleep Medicine, Endocrinology, Rheumatology, Diets and Nutritions, Allergy / Immunology |
| Gastroenterology | Gastroenterology |
| Neurology | Neurology, Neurosurgery |
| Urology / Nephrology | Urology, Nephrology |
| Obstetrics / Gynecology / Pediatrics | Obstetrics / Gynecology, Pediatrics - Neonatal |
| ENT / Ophthalmology | ENT - Otolaryngology, Ophthalmology, Dentistry, Cosmetic / Plastic Surgery |
| Hematology - Oncology | Hematology - Oncology, Lab Medicine - Pathology |
| Psychiatry / Psychology | Psychiatry / Psychology |

Dropped (rare, no natural home, or surgery with no routing class): Dermatology, IME-QME-Work Comp etc., Chiropractic, Speech - Language, Autopsy, Hospice - Palliative Care, Bariatrics (bariatric surgery pre/post-op records), 21 general-surgery / wound-care procedures from the unique Surgery rows (reviewed by the modelling lead, all confirmed DROP), and 7 thyroid surgery notes (dual-labelled Surgery plus Endocrinology or General Medicine).

## 11. Decisions checklist (Issue #1)

- [x] Strip leading spaces from labels
- [x] Drop document-type labels (6 labels)
- [x] Drop the Surgery label (keep subspecialty copies; re-label 28 unique rows, drop 21 general-surgery rows, all human-confirmed)
- [x] Duplicates: keep rows, force same text into the same split
- [x] Leakage: model on transcription only, drop keywords/description; label-string masking planned at modeling time
- [x] Missing text: drop 33 rows
- [x] Cross-label contradictions: keyword-rule first pass + human review
- [x] Final class list: 11 classes finalized (Section 10)
- [x] Bariatrics and thyroid surgery excluded from the mapping (surgery with no routing class)
- [x] Second-review corrections: 3 AI-flagged labels fixed after confirmation by the modelling lead
- [x] Cleaning pipeline implemented in code with counts per step (Section 12)

Clear all outputs before committing (raw note text must not enter the repo).

## 12. Cleaning pipeline

This section implements the cleaning steps in code, prints the note count after each step, and writes the final labels to `labels/final_labels.csv` (text ID + label only, no note text; that file is committed, see Issue #15).

Pipeline summary: 4,999 rows become 2,230 notes in 11 classes. Every excluded note has a recorded reason (no transcription, document type, Surgery label, general surgery, thyroid surgery, Bariatrics, no specialty home, or reviewer DROP). The 21 general-surgery texts excluded in 12.4 were reviewed by the modelling lead and confirmed as DROP. Three first-review labels were flagged by the LLM assistant and corrected after confirmation by the modelling lead, applied in 12.4.

The saved labels feed the frozen train / validation / test split (Issue #2, `RANDOM_SEED = 42`, stratified, same text within one split) and then the training notebooks (baseline, classical ML, ClinicalBERT).

In [ ]:
# --- 12.1 Final mapping and helpers ---

import hashlib

# Raw label -> final class. Raw labels not listed here have no specialty home
# and are excluded from the final dataset. Bariatrics is deliberately absent:
# its notes are bariatric surgery pre/post-op records with no routing class.
CLASS_MAP = {
    "Cardiovascular / Pulmonary": "Cardiovascular / Pulmonary",
    "Orthopedic": "Orthopedic", "Podiatry": "Orthopedic",
    "Radiology": "Radiology",
    "General Medicine": "General Medicine",
    "Pain Management": "General Medicine",
    "Physical Medicine - Rehab": "General Medicine",
    "Sleep Medicine": "General Medicine",
    "Endocrinology": "General Medicine",
    "Rheumatology": "General Medicine",
    "Diets and Nutritions": "General Medicine",
    "Allergy / Immunology": "General Medicine",
    "Gastroenterology": "Gastroenterology",
    "Neurology": "Neurology", "Neurosurgery": "Neurology",
    "Urology": "Urology / Nephrology", "Nephrology": "Urology / Nephrology",
    "Obstetrics / Gynecology": "Obstetrics / Gynecology / Pediatrics",
    "Pediatrics - Neonatal": "Obstetrics / Gynecology / Pediatrics",
    "ENT - Otolaryngology": "ENT / Ophthalmology",
    "Ophthalmology": "ENT / Ophthalmology",
    "Dentistry": "ENT / Ophthalmology",
    "Cosmetic / Plastic Surgery": "ENT / Ophthalmology",
    "Hematology - Oncology": "Hematology - Oncology",
    "Lab Medicine - Pathology": "Hematology - Oncology",
    "Psychiatry / Psychology": "Psychiatry / Psychology",
}

FINAL_CLASSES = [
    "Cardiovascular / Pulmonary", "Orthopedic", "Radiology", "General Medicine",
    "Gastroenterology", "Neurology", "Urology / Nephrology",
    "Obstetrics / Gynecology / Pediatrics", "ENT / Ophthalmology",
    "Hematology - Oncology", "Psychiatry / Psychology",
]

def text_id(normalized):
    """Stable ID for one normalized text (SHA-1 prefix, matches the review tool)."""
    return hashlib.sha1(normalized.encode()).hexdigest()[:8]

In [ ]:
# --- 12.2 Row-level filters, then deduplicate to unique texts ---
# normalize_text is defined in Section 5; DOC_TYPE_LABELS in Section 4.

work = df.copy()
work[LABEL_COL] = work[LABEL_COL].astype(str).str.strip()   # Step 1: no rows lost

print(f"Step 0  raw dataset:                              {len(work):>5} rows")

work = work[work[TEXT_COL].notna()].copy()                 # Step 2: missing text
print(f"Step 2  drop rows with no transcription:           {len(work):>5} rows")

work = work[~work[LABEL_COL].isin(DOC_TYPE_LABELS)].copy()  # Step 3: document types
print(f"Step 3  drop 6 document-type labels:               {len(work):>5} rows")

work["_norm"] = work[TEXT_COL].map(normalize_text)

surgery_rows = work[work[LABEL_COL].eq("Surgery")]
texts_with_surgery_label = set(surgery_rows["_norm"])  # used in 12.4 (thyroid rule)
work = work[work[LABEL_COL].ne("Surgery")].copy()          # Step 4: Surgery label
print(f"Step 4  drop the Surgery label:                    {len(work):>5} rows")

dup_mask = surgery_rows[TEXT_COL].isin(set(work[TEXT_COL]))
unique_surgery = surgery_rows[~dup_mask]
print(f"        Surgery rows dropped: {len(surgery_rows)} "
      f"({int(dup_mask.sum())} byte-identical duplicates of notes that stay "
      f"under their real specialty, {len(unique_surgery)} unique notes)")

# Step 5: one example per unique normalized text
text_labels = work.groupby("_norm")[LABEL_COL].apply(lambda s: sorted(set(s)))
print(f"Step 5  unique texts after dedup:                  {len(text_labels):>5} texts")

single = text_labels[text_labels.map(len) == 1]
cross  = text_labels[text_labels.map(len) > 1]
print(f"        single-label texts: {len(single)} | cross-label conflicts: {len(cross)}")

# Unique Surgery notes (texts filed only under Surgery), kept aside for Step 6
us_norm = unique_surgery["_norm"].drop_duplicates()
print(f"        unique Surgery texts: {len(us_norm)} (reviewed or excluded in Step 6)")

In [ ]:
# --- 12.3 Load the manual review results ---
# Reviewers exported CSVs with columns (text_id, final_label). They sit in
# data/ next to the dataset and are git-ignored; only the final labels file
# (labels/final_labels.csv) is committed.

review_files = sorted(DATA_DIR.glob("review_results*.csv"))
if review_files:
    review = pd.concat([pd.read_csv(f) for f in review_files], ignore_index=True)
    review = review.dropna(subset=["final_label"])
    review_lookup = dict(zip(review["text_id"], review["final_label"]))
    print(f"Review files loaded: {[f.name for f in review_files]}")
    print(f"Reviewed notes: {len(review_lookup)}")
else:
    review_lookup = None
    print("No review CSVs found in data/. The pipeline will fall back to "
          "labels/final_labels.csv in the next cell.")

In [ ]:
# --- 12.4 Assemble the final labelled dataset ---

# Second-review corrections (Oct 26): three first-review labels were
# flagged by the LLM assistant (two found by reading the 28 unique-Surgery
# assignments, one by a keyword scan of the General Medicine class) and
# confirmed by the modelling lead. The original reviewer exports stay
# unchanged as the audit record.
SECOND_REVIEW_CORRECTIONS = {
    "99ea3545": "Gastroenterology",                     # laparoscopic inguinal hernia repair
    "ac63df45": "DROP",                                 # buttock abscess I&D (general surgery)
    "38eddc9b": "Obstetrics / Gynecology / Pediatrics", # rectovaginal fistula closure
}
if review_lookup is not None:
    review_lookup.update(SECOND_REVIEW_CORRECTIONS)

# Thyroid surgery terms (full audit lists live in 12.5): thyroid surgery
# notes that reached General Medicine through Surgery + Endocrinology /
# General Medicine dual labels are dropped here, together with the Bariatrics
# notes (absent from CLASS_MAP), as surgery with no routing class.
THYROID_SURGERY_TERMS = ["thyroidectomy","hemithyroidectomy","lobectomy",
    "thyroid mass","thyroid goiter","goiter","central neck reoperation",
    "paratracheal node","neck dissection","thyroid surgery"]

if review_lookup is None:
    # Colab or fresh clone without the local review CSVs: use the committed labels.
    final_df = pd.read_csv(ROOT / "labels" / "final_labels.csv")
    print(f"No review CSVs available; loaded committed labels ({len(final_df)} notes).")
else:
    final_rows = []
    dropped = {"no specialty home": 0, "reviewer DROP": 0, "general surgery": 0,
               "thyroid surgery": 0, "unreviewed conflict": 0}

    # (a) single-label texts -> map to a final class, or exclude if no home
    for t, labs in single.items():
        cls = CLASS_MAP.get(labs[0])
        if cls is None:
            # includes Bariatrics (removed from CLASS_MAP: bariatric surgery
            # pre/post-op records with no routing class)
            dropped["no specialty home"] += 1
        elif (labs[0] in ("General Medicine", "Endocrinology")
              and t in texts_with_surgery_label
              and any(k in t for k in THYROID_SURGERY_TERMS)):
            # thyroid surgery notes that reached General Medicine through
            # Surgery + Endocrinology / General Medicine dual labels
            dropped["thyroid surgery"] += 1
        else:
            final_rows.append((text_id(t), cls))

    # (b) cross-label texts -> reviewer label (DROP = excluded)
    for t, labs in cross.items():
        tid = text_id(t)
        y = review_lookup.get(tid)
        if y is None:
            dropped["unreviewed conflict"] += 1
        elif y == "DROP":
            dropped["reviewer DROP"] += 1
        else:
            final_rows.append((tid, y))

    # (c) unique Surgery texts -> reviewer label, or general surgery excluded
    for t in us_norm:
        tid = text_id(t)
        y = review_lookup.get(tid)
        if y is None:
            dropped["general surgery"] += 1
        elif y == "DROP":
            dropped["reviewer DROP"] += 1
        else:
            final_rows.append((tid, y))

    final_df = pd.DataFrame(final_rows, columns=["text_id", "label"]).drop_duplicates("text_id")
    final_df = final_df.sort_values("text_id").reset_index(drop=True)

    print(f"Step 6  review applied: {len(review_lookup)} notes "
          f"({sum(1 for v in review_lookup.values() if v != 'DROP')} labelled, "
          f"{sum(1 for v in review_lookup.values() if v == 'DROP')} DROP)")
    print(f"Step 7  excluded during assembly: {sum(dropped.values())} texts "
          f"(by reason: {dropped})")

    # Save: IDs + labels only, no note text (course rule; labels/ is committed).
    LABELS_DIR = ROOT / "labels"
    LABELS_DIR.mkdir(exist_ok=True)
    final_df.to_csv(LABELS_DIR / "final_labels.csv", index=False)
    print(f"Saved: {LABELS_DIR / 'final_labels.csv'}")

print(f"\nFinal dataset: {len(final_df)} notes, {final_df['label'].nunique()} classes")
vc = final_df["label"].value_counts().reindex(FINAL_CLASSES).fillna(0).astype(int)
for c in FINAL_CLASSES:
    print(f"  {c:40s} {vc[c]:4d} ({vc[c]/len(final_df)*100:4.1f}%)")
print(f"  {'TOTAL':40s} {len(final_df):4d}")

In [ ]:
# --- 12.5 Keyword rules used for review suggestions (audit trail) ---
# The suggestions shown to reviewers were NOT produced by an LLM. They came
# from these deterministic keyword tables: for each note, the lowercased text
# is scanned table by table in the order below and the first specialty with a
# matching keyword wins; if nothing matches, the fallback is General Medicine.
# Measured agreement with the human labels: 48.5% (294/606). These rules are
# recorded here so the review process is reproducible.

SUGGESTION_RULES = [
    ("Radiology", ["x-ray","ct scan","mri","ultrasound","fluoroscop","angiogram",
                   "arteriogram","venogram","discogram","mammogram","sonogram",
                   "imaging","scan","c-arm"]),
    ("Orthopedic", ["fracture","arthroscop","bunion","knee","hip","shoulder","ankle",
                    "wrist","elbow","tendon","ligament","cartilage","osteotom",
                    "meniscus","rotator cuff","hallux","metatarsal","femur","tibia",
                    "fibula","humerus","clavicle","spondyl","fusion","laminectom",
                    "discectom","bone graft"]),
    ("Cardiovascular / Pulmonary", ["heart","cardiac","coronary","angioplasty","stent",
                    "aorta","aortic","vascular","artery","pulmonary","lung","thorac",
                    "cardiology","atrial","ventric","pacemaker","bypass","valve",
                    "echocardiogram"]),
    ("Gastroenterology", ["colon","colonoscop","endoscop","esophag","gastr","stomach",
                    "hepat","liver","bile","gallblad","cholang","pancreat","hernia",
                    "ercp","appendec","appendic","abdomen","rectum","anorectal",
                    "perirectal","hemorrhoid","diverticul","duoden","ileum","jejun",
                    "celiac","cholecyst","gastric"]),
    ("Neurology", ["brain","cranial","cervical spine","lumbar","nerve","seizure",
                    "epilep","headache","migraine","stroke","aneurysm","hydrocephalus",
                    "meningi","cerebro","spinal","disc","neuralg","neuropath","carpal",
                    "tremor","parkinson","craniotom","intracranial","dementia",
                    "alzheimer","lamin"]),
    ("Urology / Nephrology", ["kidney","renal","nephr","dialysis","bladder","prostate",
                    "prostat","urethr","ureter","stone","scrotal","testic","penile",
                    "circumcis","hydronephr","cystoscop","urologic"]),
    ("Obstetrics / Gynecology / Pediatrics", ["pregnan","obstetr","delivery","uterus",
                    "uterine","ovary","ovarian","cervix","fetal","neonat","pediatr",
                    "infant","newborn","placenta","endometri","fallopian","cesarean",
                    "amniot","gynecolog","c-section","prenatal"]),
    ("ENT / Ophthalmology", ["ear","hearing","audi","tympan","otitis","sinus","nasal",
                    "nose","throat","tonsil","laryng","pharyng","eye","ocular",
                    "cataract","retina","cornea","glaucoma","vision","vitrectom",
                    "strabismus","orbit","optic","otolaryng","septoplast","rhinoplast"]),
    ("Hematology - Oncology", ["cancer","carcinoma","tumor","tumour","leukem","lymphom",
                    "chemo","oncolog","malignan","metast","mastect","lumpectom",
                    "sarcoma","melanom","myelom","hodgkin","anemia","port-a-cath",
                    "port a cath","port insertion","biopsy"]),
    ("Psychiatry / Psychology", ["depression","depressive","anxiety","bipolar","schizo",
                    "suicid","psychiatr","psycholog","hallucin","delusion","adhd",
                    "mental health","mood disorder"]),
]

def suggest(desc, kw=""):
    """Rule-based suggestion used in the review tool. Returns
    (label, confidence, reason). The confidence is ALSO rule-based, not a
    model probability: low = no table matched (fallback General Medicine),
    high = exactly one table matched, medium = two or more tables matched
    (the first table in the order above wins)."""
    text = (desc + " " + kw).lower()
    matched = []
    for label, keys in SUGGESTION_RULES:
        for k in keys:
            if k in text:
                matched.append((label, k))
                break
    if not matched:
        return ("General Medicine", "low", "no keyword matched")
    labels = [m[0] for m in matched]
    if len(set(labels)) == 1:
        return (matched[0][0], "high", "keyword '" + matched[0][1] + "'")
    return (matched[0][0], "medium",
            "keyword '" + matched[0][1] + "' (also matches other tables)")

# Terms used in Step 4 to identify the 21 general-surgery notes (then confirmed
# by human review). Matched against description + keywords, case-insensitive.
GENERAL_SURGERY_TERMS = ["debridement","incision and drainage","i&d","laceration",
    "foreign body","eschar","stab wound","dressing change","wound",
    "soft tissue mass","chest wall mass","central venous","g-tube",
    "reintubat","fasciotom","pannus","hydradenitis","granuloma"]

# Thyroid surgery terms used in 12.4 (audit copy of the list defined there).
THYROID_SURGERY_TERMS_AUDIT = ["thyroidectomy","hemithyroidectomy","lobectomy",
    "thyroid mass","thyroid goiter","goiter","central neck reoperation",
    "paratracheal node","neck dissection","thyroid surgery"]

print(f"Recorded {len(SUGGESTION_RULES)} suggestion tables, "
      f"{len(GENERAL_SURGERY_TERMS)} general-surgery terms and "
      f"{len(THYROID_SURGERY_TERMS_AUDIT)} thyroid-surgery terms for the audit trail.")